# <font color=56A5EC>**Comparação de Modelos de Sobrevivência**</font>

## Análise de 7 Modelos de Sobrevivência com Validação Cruzada e Otimização de Hiperparâmetros

Este notebook implementa um pipeline automático de aprendizado de máquina que:
- Carrega datasets de treino/teste preprocessados
- Aplica codificação mista (ordinal + one-hot)
- Executa busca em grid com validação cruzada estratificada (5-fold)
- Treina 7 modelos de sobrevivência
- Computa 5 métricas de avaliação
- Exporta resultados, predições e melhores parâmetros para comparação

## <font color=56A5EC>**Fase 1: Inicialização e Carregamento de Dados**</font>

### <font color=FFB90F>**Instalações**</font>

In [31]:
%pip install scikit-survival --quiet
%pip install xgboost --quiet
%pip install lightgbm --quiet
%pip install --upgrade lifelines --quiet

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


### <font color=FFB90F>**Importações**</font>

In [32]:
# === Visualização ===
import matplotlib.pyplot as plt
%matplotlib inline

# === Manipulação ===
import numpy as np
import pandas as pd
import json
import warnings
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore")

# === Sklearn ===
from sklearn import set_config
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.base import clone

set_config(display="text")

# === Scikit-survival ===
from sksurv.util import Surv
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.ensemble import RandomSurvivalForest, GradientBoostingSurvivalAnalysis
from sksurv.metrics import (
    concordance_index_censored,
    concordance_index_ipcw,
    integrated_brier_score,
    cumulative_dynamic_auc
)

# === XGBoost ===
import xgboost as xgb

# === LightGBM ===
import lightgbm as lgb

print("✓ Imports loaded successfully")

✓ Imports loaded successfully


### <font color=FFB90F>**Carregamento de Dados e Metadados**</font>

In [33]:
# === Paths ===
BASE_DIR = Path("../data/processed")
RESULTS_DIR = Path("./results")
PARAMS_DIR = RESULTS_DIR / "params"
FIGURES_DIR = RESULTS_DIR / "figures"
METRICS_DIR = RESULTS_DIR / "metrics"
PREDS_DIR = RESULTS_DIR / "predictions"

for d in [RESULTS_DIR, PARAMS_DIR, FIGURES_DIR, METRICS_DIR, PREDS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("✓ Output directories ready")

✓ Output directories ready


In [34]:
# === Carregamento de Dados ===
# Load metadata
metadata_path = BASE_DIR / "dataset_metadata_2025.json"
with open(metadata_path, 'r', encoding='utf-8') as f:
    metadata = json.load(f)

# Load train and test datasets
df_train = pd.read_csv(BASE_DIR / "train_data_2025.csv")
df_test = pd.read_csv(BASE_DIR / "test_data_2025.csv")

print(f"✓ Train data loaded: {df_train.shape}")
print(f"✓ Test data loaded: {df_test.shape}")
print(f"✓ Metadata loaded with {len(metadata['predictor_columns'])} predictors")

# === Preparar dados para modelagem ===
# Extrair colunas de interesse
predictor_cols = metadata["predictor_columns"]
target_cols = metadata["target_columns"]

X_train = df_train[predictor_cols].copy()
X_test = df_test[predictor_cols].copy()

y_train_time = df_train["time"].astype(float).values
y_train_event = df_train["event"].astype(bool).values

y_test_time = df_test["time"].astype(float).values
y_test_event = df_test["event"].astype(bool).values

# Correct survival targets
y_train_surv = Surv.from_arrays(event=y_train_event, time=y_train_time)
y_test_surv = Surv.from_arrays(event=y_test_event, time=y_test_time)

print("✓ Survival targets created")
print("Train events:", y_train_event.sum(), "/", len(y_train_event))
print("Test events:", y_test_event.sum(), "/", len(y_test_event))


✓ Train data loaded: (26131, 17)
✓ Test data loaded: (6533, 17)
✓ Metadata loaded with 13 predictors
✓ Survival targets created
Train events: 14512 / 26131
Test events: 3628 / 6533


## <font color=56A5EC>**Fase 2: Pré-processamento e Pipeline de Codificação**</font>

<font color=#888B90>Preparação de colunas, transformação e codificação para alimentar os modelos da Fase 3.</font>

In [35]:
# === Metadata-driven columns ===
predictor_cols = metadata["predictor_columns"]
target_cols = metadata["target_columns"]

# Numeric columns from metadata or inferred
numeric_cols = metadata.get(
    "numeric_columns",
    [c for c in predictor_cols if pd.api.types.is_numeric_dtype(df_train[c])]
)

# Explicit categorical split
ordinal_cols = ["ECGRUP"] if "ECGRUP" in predictor_cols else []
onehot_cols = [c for c in ["TOPO", "MORFO_CAT"] if c in predictor_cols]

# Extra categoricals from metadata
extra_cats = [
    c for c in metadata.get("categorical_columns", [])
    if c not in ordinal_cols and c not in onehot_cols
]
onehot_cols = onehot_cols + [c for c in extra_cats if c not in onehot_cols]

# Remove categoricals from numeric list if needed
numeric_cols = [c for c in numeric_cols if c not in ordinal_cols + onehot_cols]

print("Predictors:", predictor_cols)
print("Numeric:", numeric_cols)
print("Ordinal:", ordinal_cols)
print("One-hot:", onehot_cols)

Predictors: ['ESCOLARI', 'IDADE', 'SEXO', 'CATEATEND', 'DIAGPREV', 'TOPO', 'ECGRUP', 'HABILIT', 'nDRS', 'CONSDIAG_CAT', 'TRATCONS_CAT', 'DIAGTRAT_CAT', 'MORFO_CAT']
Numeric: ['ESCOLARI', 'IDADE', 'SEXO', 'CATEATEND', 'DIAGPREV', 'HABILIT', 'nDRS', 'CONSDIAG_CAT', 'TRATCONS_CAT', 'DIAGTRAT_CAT']
Ordinal: ['ECGRUP']
One-hot: ['TOPO', 'MORFO_CAT']


In [36]:
# === CÉLULA DESATIVADA (duplicada) ===
# Esta célula repetia a definição de colunas da célula anterior.
# Mantida apenas para histórico do notebook.
print("Célula duplicada desativada. Use a célula anterior para definição de colunas.")

Célula duplicada desativada. Use a célula anterior para definição de colunas.


In [37]:
transformers = []

if numeric_cols:
    transformers.append(("num", StandardScaler(), numeric_cols))

if ordinal_cols:
    transformers.append((
        "ord",
        OrdinalEncoder(
            categories=[["I", "II", "III", "IV"]],
            handle_unknown="use_encoded_value",
            unknown_value=-1
        ),
        ordinal_cols
    ))

if onehot_cols:
    transformers.append((
        "cat",
        OneHotEncoder(
            handle_unknown="ignore",
            drop="first",
            sparse_output=False
        ),
        onehot_cols
    ))

preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder="drop"
)

print("✓ Preprocessor created")

✓ Preprocessor created


## <font color=56A5EC>**Fase 2.1: Funções Auxiliares de Features e Métricas**</font>

<font color=#888B90>Funções utilitárias usadas na fase de modelagem e avaliação rápida.</font>

In [38]:
def get_feature_names_from_preprocessor(preprocessor, numeric_cols, ordinal_cols, onehot_cols):
    feature_names = []

    if numeric_cols:
        feature_names.extend(numeric_cols)

    if ordinal_cols:
        feature_names.extend(ordinal_cols)

    if onehot_cols:
        ohe = preprocessor.named_transformers_["cat"]
        feature_names.extend(ohe.get_feature_names_out(onehot_cols).tolist())

    return feature_names


def cindex_scorer(estimator, X, y):
    risk_scores = estimator.predict(X)
    risk_scores = np.asarray(risk_scores).reshape(-1)
    return concordance_index_censored(y["event"], y["time"], risk_scores)[0]


def safe_predict(model, X):
    pred = model.predict(X)
    return np.asarray(pred).reshape(-1)


def evaluate_risk_model(model, X_train_ref, y_train_ref, X_test_eval, y_test_eval, model_name):
    """
    For models that already output something interpretable as risk score
    (higher value = higher risk / shorter survival).
    """
    metrics = {}

    risk_scores = safe_predict(model, X_test_eval)

    metrics["C_Index"] = concordance_index_censored(
        y_test_eval["event"], y_test_eval["time"], risk_scores
    )[0]

    try:
        metrics["C_Index_IPCW"] = concordance_index_ipcw(
            y_train_ref, y_test_eval, risk_scores
        )[0]
    except Exception:
        metrics["C_Index_IPCW"] = np.nan

    try:
        eval_times = np.array([12, 24, 36, 48, 60], dtype=float)
        eval_times = eval_times[
            (eval_times > y_test_eval["time"].min()) &
            (eval_times < y_test_eval["time"].max())
        ]

        auc_vals, mean_auc = cumulative_dynamic_auc(
            y_train_ref, y_test_eval, risk_scores, eval_times
        )
        metrics["Mean_Time_AUC"] = float(np.nanmean(auc_vals))
    except Exception:
        metrics["Mean_Time_AUC"] = np.nan

    metrics["IBS"] = np.nan
    metrics["Model_Type"] = "risk_score"

    preds_df = pd.DataFrame({
        "actual_time": y_test_eval["time"],
        "actual_event": y_test_eval["event"].astype(int),
        f"{model_name}_risk_score": risk_scores
    })

    return metrics, preds_df


def evaluate_time_regression_model(model, X_train_ref, y_train_ref, X_test_eval, y_test_eval, model_name):
    """
    For models that predict survival time directly.
    Convert predicted time into risk score by multiplying by -1.
    Higher risk should mean shorter predicted survival.
    """
    metrics = {}

    predicted_time = safe_predict(model, X_test_eval)
    risk_scores = -predicted_time

    metrics["C_Index"] = concordance_index_censored(
        y_test_eval["event"], y_test_eval["time"], risk_scores
    )[0]

    try:
        metrics["C_Index_IPCW"] = concordance_index_ipcw(
            y_train_ref, y_test_eval, risk_scores
        )[0]
    except Exception:
        metrics["C_Index_IPCW"] = np.nan

    try:
        eval_times = np.array([12, 24, 36, 48, 60], dtype=float)
        eval_times = eval_times[
            (eval_times > y_test_eval["time"].min()) &
            (eval_times < y_test_eval["time"].max())
        ]

        auc_vals, mean_auc = cumulative_dynamic_auc(
            y_train_ref, y_test_eval, risk_scores, eval_times
        )
        metrics["Mean_Time_AUC"] = float(np.nanmean(auc_vals))
    except Exception:
        metrics["Mean_Time_AUC"] = np.nan

    metrics["IBS"] = np.nan
    metrics["Model_Type"] = "time_regression_baseline"

    preds_df = pd.DataFrame({
        "actual_time": y_test_eval["time"],
        "actual_event": y_test_eval["event"].astype(int),
        f"{model_name}_predicted_time": predicted_time,
        f"{model_name}_risk_score": risk_scores
    })

    return metrics, preds_df


def evaluate_survival_function_model(model, X_train_ref, y_train_ref, X_test_eval, y_test_eval, model_name):
    metrics = {}

    risk_scores = safe_predict(model, X_test_eval)

    metrics["C_Index"] = concordance_index_censored(
        y_test_eval["event"], y_test_eval["time"], risk_scores
    )[0]

    try:
        metrics["C_Index_IPCW"] = concordance_index_ipcw(
            y_train_ref, y_test_eval, risk_scores
        )[0]
    except Exception:
        metrics["C_Index_IPCW"] = np.nan

    try:
        eval_times = np.array([12, 24, 36, 48, 60], dtype=float)
        eval_times = eval_times[
            (eval_times > y_test_eval["time"].min()) &
            (eval_times < y_test_eval["time"].max())
        ]

        surv_fns = model.predict_survival_function(X_test_eval)
        surv_probs = np.row_stack([[fn(t) for t in eval_times] for fn in surv_fns])

        metrics["IBS"] = integrated_brier_score(
            y_train_ref, y_test_eval, surv_probs, eval_times
        )

        auc_vals, mean_auc = cumulative_dynamic_auc(
            y_train_ref, y_test_eval, risk_scores, eval_times
        )
        metrics["Mean_Time_AUC"] = float(np.nanmean(auc_vals))

    except Exception:
        metrics["IBS"] = np.nan
        metrics["Mean_Time_AUC"] = np.nan

    metrics["Model_Type"] = "survival_function"

    preds_df = pd.DataFrame({
        "actual_time": y_test_eval["time"],
        "actual_event": y_test_eval["event"].astype(int),
        f"{model_name}_risk_score": risk_scores
    })

    return metrics, preds_df

## <font color=56A5EC>**Fase 3: Modelagem (Configuração + Treino Rápido)**</font>

<font color=#888B90>Esta fase concentra definição dos modelos, estratégia de validação e treino inicial para inspeção estrutural.</font>

In [39]:
# === Fase 3.1: Configuração dos modelos de sobrevivência nativos ===
models_config = {
    "Cox_PH": {
        "type": "survival_native",
        "pipeline": Pipeline([
            ("prep", preprocessor),
            ("model", CoxPHSurvivalAnalysis())
        ]),
        "params": {}
    },
    "Random_Survival_Forest": {
        "type": "survival_native",
        "pipeline": Pipeline([
            ("prep", preprocessor),
            ("model", RandomSurvivalForest(
                n_estimators=200,
                min_samples_split=10,
                min_samples_leaf=5,
                random_state=42,
                n_jobs=-1
            ))
        ]),
        "params": {}
    },
    "Gradient_Boosting_Survival": {
        "type": "survival_native",
        "pipeline": Pipeline([
            ("prep", preprocessor),
            ("model", GradientBoostingSurvivalAnalysis(
                learning_rate=0.05,
                n_estimators=100,
                random_state=42
            ))
        ]),
        "params": {}
    }
}

print("✓ Native survival models configured")
print("Modelos:", ", ".join(models_config.keys()))

✓ Native survival models configured
Modelos: Cox_PH, Random_Survival_Forest, Gradient_Boosting_Survival


In [40]:
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

feature_names_encoded = get_feature_names_from_preprocessor(
    preprocessor, numeric_cols, ordinal_cols, onehot_cols
)

X_train_encoded = pd.DataFrame(
    X_train_encoded,
    columns=feature_names_encoded,
    index=X_train.index
)

X_test_encoded = pd.DataFrame(
    X_test_encoded,
    columns=feature_names_encoded,
    index=X_test.index
)

# LightGBM prefers clean feature names
X_train_encoded.columns = [str(c).replace(" ", "_") for c in X_train_encoded.columns]
X_test_encoded.columns = [str(c).replace(" ", "_") for c in X_test_encoded.columns]

print("Encoded train shape:", X_train_encoded.shape)
print("Encoded test shape:", X_test_encoded.shape)
print("✓ Feature names cleaned for LightGBM")

Encoded train shape: (26131, 25)
Encoded test shape: (6533, 25)
✓ Feature names cleaned for LightGBM


In [41]:
# NOTE:
# XGBoost_Cox and XGBoost_AFT are included here only for structural validation.
# Their final implementations should be refined later to better handle censoring.

extra_models = {
    "XGBoost_Cox": {
        "type": "risk_score",
        "model": xgb.XGBRegressor(
            objective="survival:cox",
            n_estimators=150,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            random_state=42
        ),
        # "params": {
        #     "n_estimators": [100, 200],
        #     "max_depth": [3, 5],
        #     "learning_rate": [0.03, 0.05, 0.1]
        # }
        "params": {}
    },

    "XGBoost_AFT": {
        "type": "aft_quick_check",
        "model": xgb.XGBRegressor(
            objective="survival:aft",
            eval_metric="aft-nloglik",
            n_estimators=150,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            aft_loss_distribution="normal",
            aft_loss_distribution_scale=1.0,
            random_state=42
        ),
        # "params": {
        #     "n_estimators": [100, 200],
        #     "max_depth": [3, 5],
        #     "learning_rate": [0.03, 0.05, 0.1],
        #     "aft_loss_distribution_scale": [0.5, 1.0, 2.0]
        # }
        "params": {}
    },

    "LightGBM_Regression": {
        "type": "regression_baseline",
        "model": lgb.LGBMRegressor(
            objective="regression",
            n_estimators=150,
            max_depth=5,
            learning_rate=0.05,
            num_leaves=31,
            random_state=42
        ),
        # "params": {
        #     "n_estimators": [100, 200],
        #     "max_depth": [3, 5, 10],
        #     "learning_rate": [0.03, 0.05, 0.1],
        #     "num_leaves": [15, 31, 63]
        # }
        "params": {}
    }
}

print("✓ Complementary models configured")
print("⚠ XGBoost_Cox and XGBoost_AFT are provisional structural models")
print("⚠ LightGBM_Regression is an auxiliary baseline, not a native survival model")

✓ Complementary models configured
⚠ XGBoost_Cox and XGBoost_AFT are provisional structural models
⚠ LightGBM_Regression is an auxiliary baseline, not a native survival model


In [42]:
# Prepared for future tuning if needed
cv_strategy = KFold(n_splits=5, shuffle=True, random_state=42)

print("✓ CV strategy prepared for future tuning")
print("⚠ Hyperparameter search is currently disabled for fast validation")

✓ CV strategy prepared for future tuning
⚠ Hyperparameter search is currently disabled for fast validation


In [43]:
# === Fase 3.4: Treino dos modelos nativos (SEM busca de hiperparâmetros) ===
best_models = {}
best_params_summary = {}

print("=" * 80)
print("TRAINING NATIVE SURVIVAL MODELS (NO HYPERPARAMETER SEARCH)")
print("=" * 80)

# Bloco de GridSearchCV desativado para testes rápidos.
# Para reativar no futuro, reintroduza o bloco if param_grid com GridSearchCV.
for model_name, config in models_config.items():
    pipe = clone(config["pipeline"])

    print(f"\n{model_name}")

    pipe.fit(X_train, y_train_surv)
    best_models[model_name] = pipe
    best_params_summary[model_name] = {}
    print("  ✓ fitted with default / fast parameters")

TRAINING NATIVE SURVIVAL MODELS (NO HYPERPARAMETER SEARCH)

Cox_PH
  ✓ fitted with default / fast parameters

Random_Survival_Forest
  ✓ fitted with default / fast parameters

Gradient_Boosting_Survival
  ✓ fitted with default / fast parameters


In [44]:
# === Fase 3.5: Treino dos modelos complementares ===
y_train_time_float = y_train_time.astype(float)
y_test_time_float = y_test_time.astype(float)

# AFT bounds (prepared for future refinement if needed)
y_lower = y_train_time_float.copy()
y_upper = y_train_time_float.copy()
y_upper[~y_train_event] = np.inf

print("=" * 80)
print("TRAINING COMPLEMENTARY MODELS")
print("=" * 80)

for model_name, config in extra_models.items():
    print(f"\n{model_name}")
    model = clone(config["model"])

    try:
        if config["type"] == "aft_quick_check":
            model.fit(X_train_encoded, y_train_time_float, verbose=False)
        elif config["type"] == "risk_score":
            model.fit(X_train_encoded, y_train_time_float, verbose=False)
        else:
            model.fit(X_train_encoded, y_train_time_float)

        best_models[model_name] = model
        best_params_summary[model_name] = {}
        print("  ✓ fitted")

    except Exception as e:
        print("  ✗ error:", e)

TRAINING COMPLEMENTARY MODELS

XGBoost_Cox
  ✓ fitted

XGBoost_AFT
  ✗ error: [11:00:40] C:\actions-runner\_work\xgboost\xgboost\src\objective\aft_obj.cu:71: Check failed: info.labels_lower_bound_.Size() == ndata (0 vs. 26131) : 

LightGBM_Regression
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001428 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 164
[LightGBM] [Info] Number of data points in the train set: 26131, number of used features: 25
[LightGBM] [Info] Start training from score 35.822471
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further 

In [45]:
# OPTIONAL BLOCK — use this later if you want a more correct AFT implementation

# dtrain = xgb.DMatrix(X_train_encoded)
# dtest = xgb.DMatrix(X_test_encoded)

# dtrain.set_float_info("label_lower_bound", y_lower)
# dtrain.set_float_info("label_upper_bound", y_upper)

# aft_params = {
#     "objective": "survival:aft",
#     "eval_metric": "aft-nloglik",
#     "aft_loss_distribution": "normal",
#     "aft_loss_distribution_scale": 1.0,
#     "learning_rate": 0.05,
#     "max_depth": 4,
#     "verbosity": 0
# }

# aft_model = xgb.train(
#     params=aft_params,
#     dtrain=dtrain,
#     num_boost_round=150
# )

# best_models["XGBoost_AFT"] = aft_model
# best_params_summary["XGBoost_AFT"] = aft_params

In [46]:
metrics_results = {}
predictions_frames = []

print("=" * 80)
print("EVALUATION")
print("=" * 80)

for model_name, model in best_models.items():
    print(f"\nEvaluating {model_name}...")

    try:
        if model_name in ["Cox_PH", "Random_Survival_Forest", "Gradient_Boosting_Survival"]:
            metrics, preds_df = evaluate_survival_function_model(
                model, X_train, y_train_surv, X_test, y_test_surv, model_name
            )

        elif model_name in ["XGBoost_Cox"]:
            metrics, preds_df = evaluate_risk_model(
                model, X_train_encoded, y_train_surv, X_test_encoded, y_test_surv, model_name
            )

        elif model_name in ["LightGBM_Regression"]:
            metrics, preds_df = evaluate_time_regression_model(
                model, X_train_encoded, y_train_surv, X_test_encoded, y_test_surv, model_name
            )

        # XGBoost_AFT stays out until a proper censoring-aware implementation is used
        elif model_name in ["XGBoost_AFT"]:
            print("  skipped: provisional model without final censoring-aware implementation")
            continue

        else:
            continue

        metrics_results[model_name] = metrics
        predictions_frames.append(preds_df)

        print("  C-Index:", round(metrics["C_Index"], 4))
        print("  IPCW C-Index:", round(metrics["C_Index_IPCW"], 4) if pd.notna(metrics["C_Index_IPCW"]) else np.nan)
        print("  IBS:", round(metrics["IBS"], 4) if pd.notna(metrics["IBS"]) else np.nan)
        print("  Mean Time AUC:", round(metrics["Mean_Time_AUC"], 4) if pd.notna(metrics["Mean_Time_AUC"]) else np.nan)

    except Exception as e:
        print("  ✗ evaluation error:", e)

EVALUATION

Evaluating Cox_PH...
  C-Index: 0.7098
  IPCW C-Index: 0.7086
  IBS: 0.1862
  Mean Time AUC: 0.7651

Evaluating Random_Survival_Forest...
  C-Index: 0.7375
  IPCW C-Index: 0.7357
  IBS: 0.1728
  Mean Time AUC: 0.796

Evaluating Gradient_Boosting_Survival...
  C-Index: 0.7272
  IPCW C-Index: 0.7255
  IBS: 0.1786
  Mean Time AUC: 0.7829

Evaluating XGBoost_Cox...
  C-Index: 0.7381
  IPCW C-Index: 0.7363
  IBS: nan
  Mean Time AUC: 0.7963

Evaluating LightGBM_Regression...
  C-Index: 0.7381
  IPCW C-Index: 0.7363
  IBS: nan
  Mean Time AUC: 0.7962


In [47]:
df_metrics = pd.DataFrame(metrics_results).T

df_metrics["Composite_Score"] = (
    df_metrics["C_Index"].fillna(0) * 0.35 +
    df_metrics["C_Index_IPCW"].fillna(0) * 0.35 +
    df_metrics["Mean_Time_AUC"].fillna(0) * 0.20 +
    (1 - df_metrics["IBS"].clip(0, 1).fillna(1)) * 0.10
)

df_metrics = df_metrics.sort_values("Composite_Score", ascending=False).round(4)

print("=" * 80)
print("FINAL METRICS TABLE")
print("=" * 80)
display(df_metrics)

df_metrics.to_csv(METRICS_DIR / "evaluation_metrics_2025.csv")
print("✓ Metrics saved")

FINAL METRICS TABLE


,C_Index,C_Index_IPCW,IBS,Mean_Time_AUC,Model_Type,Composite_Score
Random_Survival_Forest,0.737535,0.735744,0.172815,0.796023,survival_function,0.7576
Gradient_Boosting_Survival,0.727169,0.725514,0.178624,0.782863,survival_function,0.7471
Cox_PH,0.709826,0.70857,0.186249,0.765099,survival_function,0.7308
LightGBM_Regression,0.7381,0.73633,NaN,0.79617,time_regression_baseline,0.6753
XGBoost_Cox,0.738066,0.736277,NaN,0.796253,risk_score,0.6753


✓ Metrics saved


In [48]:
predictions_merged = pd.DataFrame({
    "actual_time": y_test_time,
    "actual_event": y_test_event.astype(int)
})

for dfp in predictions_frames:
    score_cols = [c for c in dfp.columns if c.endswith("_risk_score")]
    for c in score_cols:
        predictions_merged[c] = dfp[c].values

predictions_merged.to_csv(PREDS_DIR / "test_predictions_2025.csv", index=False)
print("✓ Predictions saved")
print(predictions_merged.head())

✓ Predictions saved
   actual_time  actual_event  Cox_PH_risk_score  \
0          1.0             0           1.301150   
1         23.0             1           0.399941   
2         35.0             1           0.401345   
3         13.0             1           2.371916   
4         10.0             1           2.432373   

   Random_Survival_Forest_risk_score  Gradient_Boosting_Survival_risk_score  \
0                          46.655759                              -0.343005   
1                          24.330275                              -0.606321   
2                          17.038694                              -0.621277   
3                          97.188579                               0.899613   
4                          71.078438                               0.935029   

   XGBoost_Cox_risk_score  LightGBM_Regression_risk_score  
0                9.658071                      -28.331142  
1                7.861192                      -43.941961  
2                5

In [49]:
print("=" * 80)
print("PIPELINE FINISHED")
print("=" * 80)

print("\nTrain shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Encoded features:", X_train_encoded.shape[1])

if not df_metrics.empty:
    print("\nBest model:", df_metrics.index[0])
    print("Best composite score:", round(df_metrics["Composite_Score"].iloc[0], 4))

print("\nImportant notes:")
print("- Hyperparameter tuning was disabled for fast validation")
print("- Cox PH, RSF, and GBSA are the main native survival models in this run")
print("- XGBoost_Cox and XGBoost_AFT are provisional structural implementations")
print("- LightGBM_Regression is only an auxiliary baseline")

print("\nSaved files:")
print("-", METRICS_DIR / "evaluation_metrics_2025.csv")
print("-", PREDS_DIR / "test_predictions_2025.csv")
print("-", PARAMS_DIR / "hyperparameters_test_2025.json")
print("-", FIGURES_DIR / "cindex_ipcw_comparison_2025.png")
print("-", RESULTS_DIR / "model_comparison_summary_2025.json")

PIPELINE FINISHED

Train shape: (26131, 13)
Test shape: (6533, 13)
Encoded features: 25

Best model: Random_Survival_Forest
Best composite score: 0.7576

Important notes:
- Hyperparameter tuning was disabled for fast validation
- Cox PH, RSF, and GBSA are the main native survival models in this run
- XGBoost_Cox and XGBoost_AFT are provisional structural implementations
- LightGBM_Regression is only an auxiliary baseline

Saved files:
- results\metrics\evaluation_metrics_2025.csv
- results\predictions\test_predictions_2025.csv
- results\params\hyperparameters_test_2025.json
- results\figures\cindex_ipcw_comparison_2025.png
- results\model_comparison_summary_2025.json


In [50]:
# Fase desativada
print("Bloco de exportação desativado. Fluxo ativo: Fase 1 a Fase 3.")

Bloco de exportação desativado. Fluxo ativo: Fase 1 a Fase 3.


In [51]:
# Fase desativada
print("Bloco de visualização final desativado. Fluxo ativo: Fase 1 a Fase 3.")

Bloco de visualização final desativado. Fluxo ativo: Fase 1 a Fase 3.


In [52]:
# Fase desativada
print("Bloco de resumo final desativado. Fluxo ativo: Fase 1 a Fase 3.")

Bloco de resumo final desativado. Fluxo ativo: Fase 1 a Fase 3.


In [53]:
# Fase desativada
print("Bloco de finalização desativado. Fluxo ativo: Fase 1 a Fase 3.")

Bloco de finalização desativado. Fluxo ativo: Fase 1 a Fase 3.


## <font color=888B90>**Fase 4: Desativada neste notebook**</font>

<font color=#888B90>Bloco mantido apenas para referência histórica. O fluxo ativo termina na Fase 3.</font>

In [54]:
# Fase desativada
print("Fase 4 desativada. Fluxo ativo: Fase 1 a Fase 3.")

Fase 4 desativada. Fluxo ativo: Fase 1 a Fase 3.


In [55]:
# Fase desativada
print("Fase 4 desativada. Fluxo ativo: Fase 1 a Fase 3.")

Fase 4 desativada. Fluxo ativo: Fase 1 a Fase 3.


## <font color=888B90>**Fase 5: Desativada neste notebook**</font>

In [56]:
# Fase desativada
print("Fase 5 desativada. Fluxo ativo: Fase 1 a Fase 3.")

Fase 5 desativada. Fluxo ativo: Fase 1 a Fase 3.


## <font color=888B90>**Fase 6: Desativada neste notebook**</font>

In [57]:
# Fase desativada
print("Fase 6 desativada. Fluxo ativo: Fase 1 a Fase 3.")

Fase 6 desativada. Fluxo ativo: Fase 1 a Fase 3.


In [58]:
# Fase desativada
print("Fase 6 desativada. Fluxo ativo: Fase 1 a Fase 3.")

Fase 6 desativada. Fluxo ativo: Fase 1 a Fase 3.


## <font color=888B90>**Fase 7: Desativada neste notebook**</font>

In [59]:
# Fase desativada
print("Fase 7 desativada. Fluxo ativo: Fase 1 a Fase 3.")

Fase 7 desativada. Fluxo ativo: Fase 1 a Fase 3.


## <font color=888B90>**Fase 8: Desativada neste notebook**</font>

In [60]:
# Fase desativada
print("Fase 8 desativada. Fluxo ativo: Fase 1 a Fase 3.")

Fase 8 desativada. Fluxo ativo: Fase 1 a Fase 3.
